In [ ]:
%pip install crumpy

## Using `CircuitWidget`

To get started with `CircuitWidget`, import it from the `crumpy` package:

In [ ]:
from crumpy import CircuitWidget

### Basic Usage

The goal of CrumPy's `CircuitWidget` is to utilize the circuit visualization of [Crumble](https://algassert.com/crumble), a stabilizer circuit visualizer and sub-project of [Stim](https://github.com/quantumlib/Stim), to provide a convenient way to display quantum circuit timelines (in particular, with Pauli propagation) in a Jupyter notebook.

`CircuitWidget` takes a [Stim](https://github.com/quantumlib/Stim) circuit specification in the form of a Python string:

In [ ]:
stim_circuit = """
H 0
CNOT 0 2
SWAP 0 1
M 1
"""

widg = CircuitWidget(stim=stim_circuit)
widg

It's also possible to update the circuit of an existing `CircuitWidget`. Running the cell below will update the output of the previous cell to reflect the new circuit:

In [ ]:
new_circuit = """
CX 0 2
TICK
CY 1 2
TICK
CZ 0 1
"""

widg.stim = new_circuit

`CircuitWidget` also has configuration options that change the appearance of the circuit. Notice again how the above output is updated:

In [ ]:
widg.indentCircuitLines = False
widg.curveConnectors = False

# Can also specify at construction:
# CircuitWidget(stim=..., indentCircuitLines=False, curveConnectors=False)

### Propagating Paulis

A useful feature of Crumble (and CrumPy) is the ability to propagate Paulis through a quantum circuit. From the [Crumble docs](https://github.com/quantumlib/Stim/tree/main/glue/crumble#readme):
> Propagating Paulis is done by placing markers to indicate where to add terms.
> Each marker has a type (X, Y, or Z) and an index (0-9) indicating which
> indexed Pauli product the marker is modifying.

Define a Pauli X, Y, or Z marker with the `#!pragma MARK_` instruction:


In [ ]:
x_error_on_qubit_3 = "#!pragma MARKX(0) 3"
z_error_on_qubit_2 = "MARKZ(0) 2"  # Valid for use with CrumPy, but not with stim.Circuit - see below

#### Legend:
<img src="data:image/svg+xml;utf8,<svg xmlns='http://www.w3.org/2000/svg' width='140' height='46'><rect width='100%' height='100%' fill='white'/><text x='35' y='30' text-anchor='end' dominant-baseline='middle' fill='black'>Pauli</text><rect x='38' y='16' width='28' height='28' fill='white' stroke='black'></rect><text x='52' y='14' text-anchor='middle' dominant-baseline='auto' fill='black'>X</text><polygon points='56.8,22 52,33.2 47.2,22' fill='red'></polygon><line x1='52' y1='30' x2='66' y2='30' stroke='red' stroke-width='2'></line><rect x='70' y='16' width='28' height='28' fill='white' stroke='black'></rect><text x='84' y='14' text-anchor='middle' dominant-baseline='auto' fill='black'>Y</text><polygon points='88.8,22 84,33.2 79.2,22' fill='green'></polygon><line x1='84' y1='30' x2='98' y2='30' stroke='green' stroke-width='2'></line><rect x='102' y='16' width='28' height='28' fill='white' stroke='black'></rect><text x='116' y='14' text-anchor='middle' dominant-baseline='auto' fill='black'>Z</text><polygon points='120.8,22 116,33.2 111.2,22' fill='blue'></polygon><line x1='116' y1='30' x2='130' y2='30' stroke='blue' stroke-width='2'></line></svg>"
alt="red Pauli X, green Pauli Y, blue Pauli Z markers"    
/>
#### Example: `MARKX` Propagation

In [ ]:
circuit_with_mark = """
#!pragma MARKX(0) 0
TICK
CNOT 0 1
TICK
H 0
"""

CircuitWidget(stim=circuit_with_mark)

Notice how the red Pauli X marker defined on qubit 0 propagates through the CNOT, resulting in X markers on both qubits. Then, when the X marker on qubit 0 encounters the H gate, it is converted into a blue Pauli Z marker.

#### Note: `stim.Circuit` Compatibility

Some instructions used in Crumble (and therefore CrumPy) are not considered standard Stim notation. This includes `MARK_`, `POLYGON`, and `ERR` instructions.

For example, consider the circuit specification below which omits the '`#!pragma `' preceding its `MARKZ` instruction. Attempting to create a `stim.Circuit` will cause an error:

In [ ]:
%pip install stim

In [ ]:
import stim

no_pragma = """
MARKZ(0) 0
Y 1
TICK
CZ 0 1
"""

stim.Circuit(no_pragma)

Including the '`#!pragma `' avoids this issue. The `MARKZ` instruction will not show up in a `stim.Circuit` (the '`#`' in '`#!pragma `' comments out the line):

In [ ]:
with_pragma = """
#!pragma MARKZ(0) 0
Y 1
TICK
CZ 0 1
"""

ok_circuit = stim.Circuit(with_pragma)
ok_circuit

Even though there is no `MARKZ`, we now have a valid `stim.Circuit` which can easily be used with `CircuitWidget`:

In [ ]:
CircuitWidget(stim=str(ok_circuit))

### Composing with Other Widgets

The ability to programmatically update a `CircuitWidget`'s `stim` attribute (also its config options) provides an opportunity for more advanced interaction with the circuit visualization. In particular, we can utilize existing widgets from the `ipywidgets` package to create more interactive experiences.

The `traitlets` package used behind the scenes of both `CircuitWidget` and the widgets of `ipywidgets` allows for the composition of multiple widgets through functions like `link` and `dlink`. See the [`ipywidgets` documentation](https://ipywidgets.readthedocs.io/en/latest/examples/Widget%20Events.html#linking-widgets) for more advanced uses.

In [ ]:
%pip install ipywidgets

#### Example: `IntSlider` and `dlink`

In this example, an `IntSlider` widget is used in conjunction with a `CircuitWidget` to update the displayed circuit based on the slider's value. The `ipywidget.dlink` function is used to **d**irectionally **link** the slider's `value` to the circuit's `stim` (i.e. updates to the slider's `value` will update the circuit's `stim`). Here we can use the slider to see how a Pauli X marker acts when faced with consecutive H gates:

In [ ]:
import ipywidgets

def sliderToCircuit(slider_val):
    base_circuit = """
    #!pragma MARKX(0) 0
    TICK
    CX 0 1
    """
    return base_circuit + "H 0\n" * slider_val

circuit = CircuitWidget()
slider = ipywidgets.IntSlider(description="# of H gates", value=1, min=0, max=10)

ipywidgets.dlink((slider, "value"), (circuit, "stim"), sliderToCircuit)

display(slider, circuit)

## Appendix: Converting to Stim

If `stim` isn't your quantum circuit package of choice, it may still be possible to utilize CrumPy. Circuits from packages like `cirq` or `qiskit` may be convertible to a `stim` circuit (which can be easily converted to a string for use with `CircuitWidget`).

Note that it may not always be possible to convert a circuit between formats (notably, [`stim` has a limited gate set](https://github.com/quantumlib/Stim/tree/main?tab=readme-ov-file#what-is-stim:~:text=There%20is%20no%20support%20for%20non%2DClifford%20operations%2C%20such%20as%20T%20gates%20and%20Toffoli%20gates.%20Only%20stabilizer%20operations%20are%20supported.)).

### Example: `cirq` to `stim` with `stimcirq`

In [ ]:
%pip install cirq
%pip install stimcirq

Let's create a simple `cirq` circuit we want converted to `stim` (note that importing cirq might take a minute):

In [ ]:
import cirq

q0 = cirq.LineQubit(0)
q1 = cirq.LineQubit(1)
cirq_circuit = cirq.Circuit(cirq.H(q0), cirq.CNOT(q0, q1))

cirq_circuit

We can use the `stimcirq` package (a sub-package within the Stim project) to convert to a `stim.Circuit`:

In [ ]:
import stimcirq

converted_stim = stimcirq.cirq_circuit_to_stim_circuit(cirq_circuit)
converted_stim

Note that for `CircuitWidget` compatibility, we need this circuit as a string:

In [ ]:
converted_stim_str = str(converted_stim)

CircuitWidget(stim=converted_stim_str)